# P09A — Generator data sintesis dan augmentasi

Membuat dua data latihan baru dari DGP yang diketahui: klasifikasi spalling kolom beton bertulang dengan kelas minoritas serta kapasitas momen balok W. Seluruh data sintesis; bukan hasil uji atau hukum desain.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
rng=np.random.default_rng(20261116); n=1800
width=np.clip(rng.normal(350,65,n),200,550); steel_ratio=np.clip(rng.lognormal(np.log(2.0),.42,n),.5,8)
fc=np.clip(rng.lognormal(np.log(42),.42,n),18,110); cover=np.clip(rng.normal(38,8,n),18,65)
load_ratio=np.clip(rng.beta(2.3,3.5,n),.08,.95); fire_min=np.clip(rng.gamma(3.5,24,n)+20,20,180)
# DGP logit sederhana; minority outcome sengaja jarang
logit=-3.15 + .023*(fire_min-60) + 2.1*(load_ratio-.45) + .028*(cover-35) - .012*(fc-40) + .10*(steel_ratio-2)
p=1/(1+np.exp(-logit)); spall=rng.binomial(1,p)
rc=pd.DataFrame({'column_id':[f'C{i+1:04d}' for i in range(n)],'width_mm':width,'steel_ratio_pct':steel_ratio,'fc_mpa':fc,'cover_mm':cover,'load_ratio':load_ratio,'fire_minutes':fire_min,'spalling':np.where(spall==1,'Spalling','Tidak spalling')})
# DGP mekanika untuk symbolic regression, nilai momen dalam kNm
nb=1400; Z=rng.uniform(500,4000,nb); fy=rng.uniform(250,500,nb); M=0.001*Z*fy*(1+rng.normal(0,.012,nb))
beam=pd.DataFrame({'beam_id':[f'B{i+1:04d}' for i in range(nb)],'Z_cm3':Z,'fy_mpa':fy,'moment_kNm':M})
out=Path('praktikum/bab09'); out.mkdir(parents=True,exist_ok=True)
rc.to_csv(out/'rc_spalling_synthetic.csv',index=False); beam.to_csv(out/'beam_moment_synthetic.csv',index=False)
# Orange .tab carries explicit variable types and target role so Test & Score can run without manual correction.
cols=list(rc.columns); types=['string']+['continuous']*6+['discrete']; roles=['meta']+['']*6+['class']
with open(out/'rc_spalling_orange.tab','w',encoding='utf-8',newline='') as f:
 import csv
 w=csv.writer(f,delimiter='\t'); w.writerow(cols); w.writerow(types); w.writerow(roles); w.writerows(rc.itertuples(index=False,name=None))
print('seed=20261116 | RC n=',len(rc),'| spalling rate=',round(spall.mean(),3),'| beams=',len(beam),'| saved to',out)